In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as f

# 1. Schema
schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("account_type", StringType(), True),
    StructField("account_balance", DoubleType(), True),
    StructField("account_status", StringType(), True),
    StructField("business_date", DateType(), True)
])

# 2. Source path
path = "/Volumes/workspace/default/project_for_incremental/project_incremental/"

# 3. Auto Loader
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option(
        "cloudFiles.schemaLocation",
        "/Volumes/workspace/default/project_for_incremental/schema/"
    )
    .option("header", "true")
    .option("sep", "\t")
    .schema(schema)
    .load(path)
)





In [0]:
from pyspark.sql import functions as f

# 4. Add ingestion timestamp
df_with_timestamp = df.withColumn(
    "ingestion_timestamp",
    f.current_timestamp()
)

In [0]:
# 5. Write new files to Bronze
query = (
    df_with_timestamp.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/default/project_for_incremental/bronze_checkpoint/"
    )
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("bronze_customer_data")
)